In [ ]:
import numpy as np, pandas as pd, glob
from pathlib import Path

data_dir = Path("../data/dataset")
print(sorted(p.name for p in data_dir.iterdir())[:20])

npz_files = sorted(glob.glob(str(data_dir / "**/*.npz"), recursive=True))
print(len(npz_files), "npz files")

d = np.load(npz_files[0], allow_pickle=True)
for k in d.files:
    print(k, d[k].shape, d[k].dtype)

['.DS_Store', 'Test', 'Training', 'Validation']
3300 npz files
oct_bscans (200, 200, 200) uint8
rnflt (200, 200) float64
md () float64
glaucoma () int64
tds (52,) float64
race () <U18
male () int64
hispanic () int64
language () int64
maritalstatus () int64
age () float64


In [5]:
from pathlib import Path

rows = []
for f in npz_files:
    z = np.load(f, allow_pickle=True)
    row = {k: z[k].item() for k in ["age", "male", "race", "hispanic", "language", "maritalstatus", "glaucoma", "md"]}
    row["split"] = Path(f).parent.name      # Training / Validation / Test
    row["file"] = Path(f).name
    rows.append(row)

df = pd.DataFrame(rows)
print(df["split"].value_counts())
print(df["race"].value_counts())
df.head()

split
Training      2100
Test           900
Validation     300
Name: count, dtype: int64
race
White or Caucasian           1100
Black or African American    1100
Asian                        1100
Name: count, dtype: int64


,age,male,race,hispanic,language,maritalstatus,glaucoma,md,split,file
0,60.284932,1,White or Caucasian,0,0,0,0,-0.16,Test,data_2401.npz
1,50.115068,0,White or Caucasian,1,0,1,0,0.59,Test,data_2402.npz
2,27.890411,0,White or Caucasian,-1,0,1,0,0.37,Test,data_2403.npz
3,47.621918,1,White or Caucasian,0,0,0,1,-3.68,Test,data_2404.npz
4,52.528767,1,White or Caucasian,0,0,0,0,0.05,Test,data_2405.npz


In [10]:
cat_cols = ["race", "male", "hispanic", "language", "maritalstatus"]

for col in cat_cols:
    print(f" {col}")
    print(df.groupby(col)["glaucoma"].agg(["mean", "count"]))
    print()


#Overall count and percentages of missing values pr column.
coded_cols = ["hispanic", "language", "maritalstatus"]

summary = pd.DataFrame({
    "n_missing": [(df[c] == -1).sum() for c in coded_cols],
    "pct_missing": [(df[c] == -1).mean() * 100 for c in coded_cols],
}, index=coded_cols)
print(summary.round(1))

# also check for true NaNs anywhere in the table
print(df.isna().sum())

 race
                               mean  count
race                                      
Asian                      0.481818   1100
Black or African American  0.620909   1100
White or Caucasian         0.486364   1100

 male
          mean  count
male                 
0     0.515453   1812
1     0.547043   1488

 hispanic
              mean  count
hispanic                 
-1        0.484043    188
 0        0.530909   3025
 1        0.586207     87

 language
              mean  count
language                 
-1        0.548387     31
 0        0.500521   2877
 1        0.763158     38
 2        0.740113    354

 maritalstatus
                   mean  count
maritalstatus                 
-1             0.543210     81
 0             0.515924   1884
 1             0.501584    947
 2             0.577381    168
 3             0.740331    181
 4             0.666667     39

               n_missing  pct_missing
hispanic             188          5.7
language              31          0

In [13]:
#Chack if age and martial status is connected in a way
print(df.groupby("maritalstatus")["age"].mean())

#Race by sex table
print(df.groupby(["race", "male"])["glaucoma"].agg(["mean", "count"]))

#Glaucoma rate by split and race, to see if the race gap holds in Training,
# Validation and Test.
print(df.groupby(["split", "race"])["glaucoma"].agg(["mean", "count"]))


maritalstatus
-1    59.749180
 0    61.452132
 1    50.062919
 2    63.836679
 3    77.184744
 4    61.843123
Name: age, dtype: float64
                                    mean  count
race                      male                 
Asian                     0     0.473950    595
                          1     0.491089    505
Black or African American 0     0.620690    609
                          1     0.621181    491
White or Caucasian        0     0.450658    608
                          1     0.530488    492
                                          mean  count
split      race                                      
Test       Asian                      0.480000    300
           Black or African American  0.633333    300
           White or Caucasian         0.516667    300
Training   Asian                      0.471429    700
           Black or African American  0.605714    700
           White or Caucasian         0.470000    700
Validation Asian                      0.560000  